In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns 

#load dataset
path_to_file = 'House Price Prediction Dataset.csv'
df = pd.read_csv(path_to_file, delimiter=',')
print("df :", df)
print("df.shape :", df.shape)
print("df.head(): ", df.head())

print(df.isnull().sum())

#Preprocessing
#Garage column(true/false → 1/0)
df['Garage'] = df['Garage'].map({'Yes': 1, 'No': 0, True: 1, False: 0, 'true': 1, 'false': 0})

# Text columns 
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Location']  = le.fit_transform(df['Location'].astype(str))
df['Condition'] = le.fit_transform(df['Condition'].astype(str))

#Missing values fill 
df.fillna(df.median(numeric_only=True), inplace=True)

print("\nAfter Preprocessing:")
print(df.head())


In [ ]:
#seaborn graph 
#price distribution
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(8,5))
sns.histplot(df['Price'], bins=30, color='steelblue')
plt.title('House Price Distribution')
plt.xlabel('Price')
plt.show()

#Area vs Price
plt.figure(figsize=(8,5))
sns.scatterplot(x='Area', y='Price', data=df, alpha=0.5)
plt.title('Area vs Price')
plt.show()

#correlation heatmap
plt.figure(figsize=(10,7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.show()




In [ ]:
#Train Model 
features = ['Area', 'Bedrooms', 'Bathrooms', 'Floors',
            'YearBuilt', 'Location', 'Condition', 'Garage']

X = df[features]
y = df['Price']

# Feature Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

#linear Regression
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

#Gradient Boosting
from sklearn.ensemble import GradientBoostingRegressor
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)

#Evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error 
lr_mae  = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))

gb_mae  = mean_absolute_error(y_test, gb_pred)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_pred))

print("\n Linear Regression ")
print(f"MAE  : {lr_mae:,.2f}")
print(f"RMSE : {lr_rmse:,.2f}")

print("\n Gradient Boosting ")
print(f"MAE  : {gb_mae:,.2f}")
print(f"RMSE : {gb_rmse:,.2f}")


In [ ]:
#ACTUAL vs PREDICTED PLOT
plt.figure(figsize=(8,5))
plt.scatter(y_test, lr_pred, alpha=0.5, color='blue', label='Linear Regression')
plt.scatter(y_test, gb_pred, alpha=0.5, color='green', label='Gradient Boosting')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', label='Perfect Prediction')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted House Prices')
plt.legend()
plt.show()

#feature importance 
importance = pd.Series(gb.feature_importances_, index=features)
importance = importance.sort_values(ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x=importance.values, y=importance.index, hue=importance.index, palette='viridis', legend=False)
plt.title('Feature Importance - Gradient Boosting')
plt.xlabel('Importance Score')
plt.show()